# Logistic regression: tuning and comparison with LightGBM (Phase 4)

The proposal compares two models: **class-weighted logistic regression** and **LightGBM**. Notebook 04 fitted an
untuned logistic baseline on the store features. This notebook finishes it: the logistic regression gets the **same 38
frozen inputs** as the frozen LightGBM (28 store features, the 8 `roll4` features, `weeks_since_open`, `app_score`;
decision 28 Oct 2026), the same rows, folds, seed and measures as the feature lab and notebook 08. So the only
difference between the two is the model.

| Â§ | What | Time |
|---|---|---|
| 0 | Settings (**the toggle is here**) and setup | seconds |
| 1 | The table and the 38 inputs | a few minutes |
| 2 | How the inputs are prepared for a linear model | seconds |
| 3 | The search over C (regularisation strength) | **long**, skipped when `RUN_SEARCH = False` |
| 4 | Tuned logistic vs tuned LightGBM, same folds; the onboarding value for each | 9 model fits |
| 5 | Figures: week by week, coefficients | seconds |
| 6 | Report and the settings file for `python -m models train --model logreg` | seconds |

**Toggle.** Run once with `RUN_SEARCH = True`; every finished trial is saved to `trials.csv` at once and a rerun skips
it, so a stopped kernel loses nothing. Then `RUN_SEARCH = False` only loads `best_params.json`.

**Rules kept.** Labels only through `amldata.labels()`; months 0â€“4 only (month 5 stays for calibration, months 6â€“7
stay closed); no hidden file is read. Hidden types only break results down. Class weights stay `balanced` (team
decision 9 Oct 2026). The deciding measure is the team's **weekly recall** (decision 26 Oct 2026); **catch_rules**
(flags limited to the rules' own alert count, proposal 28 Oct 2026) is reported beside it.

## 0. Settings and setup

In [ ]:
from __future__ import annotations
RUN_SEARCH = True        # False: load best_params.json from the report folder instead of searching
C_GRID = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]   # regularisation strength (smaller = stronger shrinkage); log-spaced
                                               # around scikit-learn's default 1.0, as is usual for C

import json, os, sys, time
from pathlib import Path
if os.environ.get("LOGREG_C_GRID"):           # e.g. "0.01,1" for a quick check run
    C_GRID = [float(c) for c in os.environ["LOGREG_C_GRID"].split(",")]

import numpy as np
import pandas as pd
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "amldata").is_dir():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("start this notebook from inside the aml-detection-system repo")
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import amldata
from models import feature_lab as lab
from models import logreg
from models.lgbm_baseline import FOLDS, KEY, by_week, load_table, profiles

OUT = os.environ.get("EDA_DATA", amldata.DEFAULT_OUT)
RUN = os.path.basename(os.path.normpath(OUT))
REPORT = Path(os.environ.get("EDA_REPORT", f"outputs/modelling/models/{RUN}/logreg_tuning"))
REPORT.mkdir(parents=True, exist_ok=True)
TRIALS_CSV = REPORT / "trials.csv"
LGBM_PARAMS = "outputs/modelling/models/v1_full/lgbm_tuning_step2/tuned_params.json"    # notebook 08's result
MONTHS, WEEKS, SEED = (0, 1, 2, 3, 4), (0, 26), 42
FROZEN = logreg.INPUTS
ACTIVITY = [c for c in FROZEN if c != "app_score"]
MEASURES = ["weekly_recall", "catch_rules", "acc_pr_auc", "caught_2wk", "first_caught_2wk"]
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
COLOR = {"logistic": "#2a78d6", "LightGBM": "#e34948"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
                     "font.size": 10})
show = (lambda x: display(x)) if "ipykernel" in sys.modules else print
T0 = time.time()
print(f"data {OUT} -> {REPORT}")

## 1. The table

The feature lab's table: one row per account per open week 0â€“26 (weeks after the bank closed an account dropped),
the 28 store features, the 8 adopted 4-week features, `weeks_since_open`, label `mule_by_day`, and the application
score per fold (a logistic regression on the 56 prepared application columns, out of fold for training accounts,
as in notebook 06).

In [ ]:
df, _ = load_table(OUT, MONTHS, WEEKS, adopted=True)
df = df[KEY + ["open_month", "mule_by_day"] + [c for c in ACTIVITY if c not in KEY]] \
    .sort_values(KEY, kind="stable").reset_index(drop=True)
scores = {f: s["app_score"] for f, s in lab.app_scores(OUT, SEED).items()}
prof, start = profiles(OUT, MONTHS), lab.start_weeks(OUT)
ctx = lab.rules_context(OUT)
print(f"{len(df):,} account-weeks, {int(df.mule_by_day.sum()):,} mule weeks, {df.account_id.nunique():,} accounts "
      f"({time.time() - T0:.0f}s)")

## 2. Preparing the inputs for a linear model

A tree model splits on order only; a linear model adds up `coefficient x value`, so long tails, empty values and
shares above 1 have to be handled first. `models.logreg` does it, fitted on each fold's **training months only**:

| kind | inputs | steps |
|---|---|---|
| log | counts, money, maxima, running totals, peer ratio, minutes (also over 4 weeks) | fill a gap with the median, log(1 + x), standardise |
| share | pass-through, speed and mix shares (also over 4 weeks) | fill, clip to [0, 1], standardise |
| z | the three self-vs-past z-scores | fill, clip to [-10, 10], standardise |
| std | `weeks_since_open`, `app_score` (already log-odds) | fill, standardise |
| flag | every input with a gap in the training rows | a 0/1 column `<input>__missing` |

The fitted preparation and the coefficients are plain numbers, saved as JSON with the model.

In [ ]:
kinds = pd.Series(logreg.KIND)
show(pd.DataFrame({"n": kinds.value_counts(), "inputs": {k: ", ".join(kinds.index[kinds == k]) for k in kinds.unique()}}))

## 3. The search over C

`C` is the one setting that matters for a logistic regression: it sets how strongly the coefficients are pulled
towards 0 (L2 penalty). Each value of the grid in Â§0 is fitted on the 3 folds; the best is the one with the highest
**weekly recall, mean of the 3 folds** (the team's tuning measure, as in notebook 08).

In [ ]:
def run_view(cols, C, view):
    r, oof = lab.fit_folds(df, cols, prof, start, scores, SEED, ctx,
                           make_model=lambda: logreg.MuleLogit(C=C, random_state=SEED))
    return r.assign(model="logistic", view=view, C=C), oof


done = pd.read_csv(TRIALS_CSV) if TRIALS_CSV.exists() else pd.DataFrame()
if RUN_SEARCH:
    for C in C_GRID:
        if len(done) and (np.isclose(done["C"], C)).any():
            print(f"C={C:g}: already in {TRIALS_CSV}")
            continue
        t = time.time()
        r, oof = run_view(FROZEN, C, "frozen")
        row = {"C": C, **{f"{m}_mean": r[m].mean() for m in MEASURES},
               **{f"{m}_folds": " / ".join(f"{x:.3f}" for x in r[m]) for m in ("weekly_recall", "catch_rules")},
               "weekly_recall_pooled": lab.pooled_weekly_recall(oof), "seconds": round(time.time() - t)}
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
        done.to_csv(TRIALS_CSV, index=False)
        print(f"C={C:g}: weekly recall {row['weekly_recall_mean']:.3f} ({row['weekly_recall_folds']}), catch_rules "
              f"{row['catch_rules_mean']:.3f}, PR-AUC {row['acc_pr_auc_mean']:.3f} ({row['seconds']}s)")
    best = done.loc[done["weekly_recall_mean"].idxmax()]
    best_params = {"C": float(best["C"])}
    json.dump({"params": best_params, "weekly_recall_cv": float(best["weekly_recall_mean"]), "grid": C_GRID,
               "data": OUT, "inputs": FROZEN, "fixed": {"class_weight": "balanced", "max_iter": 2000}},
              open(REPORT / "best_params.json", "w"), indent=2)
    json.dump(best_params, open(REPORT / "tuned_params.json", "w"), indent=2)      # for models train --params
else:
    best_params = json.load(open(REPORT / "best_params.json"))["params"]
show(done.sort_values("C").round(4))
print(f"best: C = {best_params['C']:g}")
if best_params["C"] in (min(C_GRID), max(C_GRID)):
    print("note: the best C is at the edge of the grid; widen C_GRID and run the search again")

## 4. Tuned logistic vs tuned LightGBM, same folds

Both models on the same folds, with and without the application score. **Activity** = the 37 inputs without
`app_score`: the gap between the two views is the value of onboarding information for each model (professor's
comment 2: same rows, same cohort, only the application left out).

In [ ]:
C = best_params["C"]
lgbm_params = lab.read_params(LGBM_PARAMS) if os.path.exists(LGBM_PARAMS) else {}
CMP_RES, CMP_OOF = REPORT / f"_cache_compare_C{C:g}_folds.parquet", REPORT / f"_cache_compare_C{C:g}_oof.parquet"
if CMP_RES.exists() and CMP_OOF.exists():                         # saved by an earlier run with the same C
    res = pd.read_parquet(CMP_RES)
    o = pd.read_parquet(CMP_OOF)
    oofs = {k: g.drop(columns=["model", "view"]) for k, g in o.groupby(["model", "view"])}
    print(f"comparison loaded from {CMP_RES.name}")
else:
    res, oofs = [], {}
    for view, cols in {"frozen": FROZEN, "activity": ACTIVITY}.items():
        t = time.time()
        r, oof = run_view(cols, C, view)
        res.append(r); oofs[("logistic", view)] = oof
        r, oof = lab.fit_folds(df, cols, prof, start, scores if "app_score" in cols else None, SEED, ctx,
                               params=lgbm_params)
        res.append(r.assign(model="LightGBM", view=view)); oofs[("LightGBM", view)] = oof
        print(f"{view}: both models fitted ({time.time() - t:.0f}s)")
    res = pd.concat(res, ignore_index=True)
    res.astype({c: str for c in res.columns if res[c].dtype == object}).to_parquet(CMP_RES, index=False)
    pd.concat([o.assign(model=k[0], view=k[1]) for k, o in oofs.items()]).to_parquet(CMP_OOF, index=False)
cmp = res.groupby(["model", "view"])[MEASURES + ["rules"]].mean()
cmp["weekly_recall_pooled"] = [lab.pooled_weekly_recall(oofs[k]) for k in cmp.index]
show(cmp.round(3))
per_fold = res.pivot_table(index=["view", "fold"], columns="model", values=["weekly_recall", "catch_rules"])
show(per_fold.round(3))
onboarding = (cmp.xs("frozen", level="view") - cmp.xs("activity", level="view"))[["weekly_recall", "catch_rules",
                                                                                  "acc_pr_auc"]]
print("value of the application score (frozen - activity):")
show(onboarding.round(3))
real_cols = ["top_1%_witting", "top_1%_unwitting", "rules_budget_witting", "rules_budget_unwitting"]
real = res[res.view == "frozen"].groupby("model")[real_cols].agg(lambda s: " / ".join(s.astype(str)))
show(real)

## 4b. Variant: interaction terms

Section 4 shows the plain logistic ranks mules higher on average each week, but its very top is filled with normal
accounts holding very little money: a speed share of 1 on a GBP 20 balance scores as high as on GBP 5,000, and a
linear model cannot learn that the share matters only with real money behind it. A standard remedy is to hand it the
product: each share times log(1 + money in) over the same window (4 terms, `logreg.INTERACTIONS`; decision 31 Oct
2026, tried as a variant). The 38 inputs stay the same. C is searched again over three values around the plain
model's best (0.01 and 0.1; C = 1 did not finish within 12 h on this laptop, and the plain model's results were flat for C from 0.01 to 10). The final logistic is whichever variant has the higher weekly recall (mean of 3 folds), the same rule
as the C search.

In [ ]:
INTER_CSV = REPORT / "trials_interactions.csv"
inter = pd.read_csv(INTER_CSV) if INTER_CSV.exists() else pd.DataFrame()
for Ci in [0.01, 0.1]:     # C = 1 ran over 12 h on this laptop (sleep); the plain model was flat for C 0.01-10
    if len(inter) and np.isclose(inter["C"], Ci).any():
        continue
    t = time.time()
    r, oof = lab.fit_folds(df, FROZEN, prof, start, scores, SEED, ctx,
                           make_model=lambda: logreg.MuleLogit(C=Ci, interactions=True, random_state=SEED))
    real_i = {k: " / ".join(r[k].astype(str)) for k in ("rules_budget_witting", "rules_budget_unwitting",
                                                       "top_1%_witting", "top_1%_unwitting")}
    row = {"C": Ci, **{f"{m}_mean": r[m].mean() for m in MEASURES},
           **{f"{m}_folds": " / ".join(f"{x:.3f}" for x in r[m]) for m in ("weekly_recall", "catch_rules")},
           "weekly_recall_pooled": lab.pooled_weekly_recall(oof), **real_i, "seconds": round(time.time() - t)}
    inter = pd.concat([inter, pd.DataFrame([row])], ignore_index=True)
    inter.to_csv(INTER_CSV, index=False)
    print(f"interactions, C={Ci:g}: weekly recall {row['weekly_recall_mean']:.3f}, catch_rules "
          f"{row['catch_rules_mean']:.3f}, PR-AUC {row['acc_pr_auc_mean']:.3f} ({row['seconds']}s)")
show(inter.round(4))
plain_best = float(done.loc[np.isclose(done["C"], C), "weekly_recall_mean"].iloc[0])
bi = inter.loc[inter["weekly_recall_mean"].idxmax()]
use_inter = bool(bi["weekly_recall_mean"] > plain_best)
final_params = {"C": float(bi["C"]) if use_inter else C, "interactions": use_inter}
json.dump(final_params, open(REPORT / "tuned_params.json", "w"), indent=2)      # for models train --params
print(f"plain C={C:g}: {plain_best:.4f} | interactions C={bi['C']:g}: {bi['weekly_recall_mean']:.4f} "
      f"-> final logistic {final_params}")

## 5. Figures

Left: weekly recall in each week since opening (validation months pooled), both models, frozen inputs. Right: the
largest coefficients of the tuned logistic regression fitted on months 0â€“3 (the last fold's training months); every
input is standardised, so sizes compare (one standard deviation multiplies the odds by e^coef).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for model in ("logistic", "LightGBM"):
    w = by_week(oofs[(model, "frozen")].assign(view=model))
    axes[0].plot(w["week"], w["recall_top_1pct"], color=COLOR[model], lw=2, marker="o", ms=3,
                 label=f"{model} (mean {w['recall_top_1pct'].mean():.3f})")
axes[0].axvspan(8, 17, color="#f0efec", lw=0, label="fakes start laundering (weeks 8-17)")
axes[0].set(xlabel="week since opening", ylabel="share of mules in the top 1%", title="Weekly recall, frozen inputs")
axes[0].legend(frameon=False, fontsize=9)

d = df.assign(app_score=df["account_id"].map(scores[len(FOLDS)]))
fit_rows = d[d.open_month.isin(FOLDS[-1][0])]
m = logreg.MuleLogit(random_state=SEED, **final_params).fit(fit_rows[FROZEN], fit_rows["mule_by_day"])
coef = m.coefficients().head(15).iloc[::-1]
axes[1].barh(coef["column"], coef["coef"], color=[COLOR["logistic"] if v > 0 else MUTED for v in coef["coef"]],
             height=0.6)
axes[1].axvline(0, color=INK, lw=1)
axes[1].set(xlabel="coefficient (per standard deviation)", title="Largest coefficients, tuned logistic")
axes[1].grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(REPORT / "01_weekly_and_coefficients.png", dpi=130)
plt.show() if "ipykernel" in sys.modules else plt.close(fig)
m.coefficients().to_csv(REPORT / "coefficients.csv", index=False)

## 6. Report

In [ ]:
res.to_csv(REPORT / "folds.csv", index=False)
cmp.to_csv(REPORT / "comparison.csv")
lines = [f"# Logistic regression: tuning and comparison with LightGBM ({RUN})", "",
         f"Data `{OUT}`, months 0-4, weeks 0-26, the feature lab's 3 folds, seed {SEED}. Inputs: the 38 frozen inputs "
         f"(activity view: 37, without `app_score`). Logistic: `models.logreg.MuleLogit`, class weights balanced, "
         f"L2 penalty, C searched over {C_GRID}; best C = {C:g} (highest weekly recall, mean of 3 folds). LightGBM: "
         f"tuned settings `{LGBM_PARAMS}`.", "",
         "## Search over C", "", done.sort_values("C").round(4).to_markdown(index=False), "",
         "## Logistic vs LightGBM (mean of 3 folds)", "", cmp.round(3).to_markdown(), "",
         "`rules` = catch_rules of the rules' own alerts on the same months.", "",
         "## Per fold", "", per_fold.round(3).to_markdown(), "",
         "## Value of the application score (frozen - activity)", "", onboarding.round(3).to_markdown(), "",
         "## Real-customer mules in the account-level top 1% (per fold)", "", real.to_markdown(), "",
         "## Variant: interaction terms (share x log money in)", "", inter.round(4).to_markdown(index=False), "",
         f"Final logistic: `{json.dumps(final_params)}` (higher weekly recall, mean of 3 folds; plain best "
         f"{plain_best:.4f}).", "",
         "![weekly and coefficients](01_weekly_and_coefficients.png)", "",
         f"Train the final model: `python -m models train --out {OUT} --model logreg --name logreg "
         f"--params {REPORT.as_posix()}/tuned_params.json`.", ""]
(REPORT / "report.md").write_text("\n".join(lines), encoding="utf-8")
print(f"-> {REPORT} ({time.time() - T0:.0f}s total)")